In [1]:
import sys
from pathlib import Path

path = Path().cwd().parent / "src"
sys.path.insert(0, str(path))

In [2]:
from dask_obj.core import *
from dask_obj.expr import *
from dask_obj.polars import *

In [3]:
import gzip
import math
import operator
from collections import Counter, deque
from copy import deepcopy
from datetime import datetime
from functools import total_ordering
from io import StringIO
from itertools import islice
from operator import attrgetter, itemgetter, methodcaller
from pathlib import Path
from typing import Iterable

import dask
import dask.array as da
import dask.bag as db
import dask.dataframe as dd
import fsspec
import glom
import numpy as np
import pandas as pd
import polars as pl
import toolz
from dask import compute, delayed, persist
from dask.delayed import delayed
from dask.distributed import Client, LocalCluster, get_client
from IPython.display import display

In [4]:
# from itables import init_notebook_mode, show
# import itables.options as opt

# opt.maxBytes = 0
# opt.maxColumns = 1000
# opt.maxRows = 1000
# init_notebook_mode(all_interactive=True)

In [5]:
# pl.Config.set_tbl_cols(1000)

In [6]:
path = "~/e/data/gharchive"
path = Path(path).expanduser()
print(path)
files = list(map(str, [*path.glob("*.json.gz")]))
files[:5]

/home/brl0/e/data/gharchive


['/home/brl0/e/data/gharchive/2023-03-01-0.json.gz',
 '/home/brl0/e/data/gharchive/2023-03-01-1.json.gz',
 '/home/brl0/e/data/gharchive/2023-03-01-10.json.gz',
 '/home/brl0/e/data/gharchive/2023-03-01-11.json.gz',
 '/home/brl0/e/data/gharchive/2023-03-01-12.json.gz']

In [7]:
file = "/home/brl0/e/data/gharchive/2023-03-01-11.json/2023-03-01-11.json"

In [8]:
e = Expr(files[0]).F(pl_read_ndjson_fsspec, lines=1000).F(unnest)
objs = DaskDelayedObjects(files[:10])
dfs = objs.map(e.eval).to_pandas()

In [9]:
e

unnest(pl_read_ndjson_fsspec('/home/brl0/e/data/gharchive/2023-03-01-0.json.gz', lines=1000))

In [10]:
results = dfs.compute()

In [11]:
pd.concat(results)

,id,type,actor_id,actor_login,actor_display_login,actor_gravatar_id,actor_url,actor_avatar_url,repo_id,repo_name,...,payload_pull_request_milestone_closed_issues,payload_pull_request_milestone_state,payload_pull_request_milestone_created_at,payload_pull_request_milestone_updated_at,payload_issue_pull_request_merged_at,payload_release_mentions_count,payload_release_mentions,payload_pull_request_head_repo_mirror_url,payload_pages,payload_issue_milestone_due_on
0,27400475723,DeleteEvent,49699333,dependabot[bot],dependabot,,https://api.github.com/users/dependabot[bot],https://avatars.githubusercontent.com/u/49699333?,146641150,tektoncd/pipeline,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,27400475733,IssuesEvent,1882561,houstonbot,houstonbot,,https://api.github.com/users/houstonbot,https://avatars.githubusercontent.com/u/1882561?,172130649,cph/SearchWithModifiers,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,27400475735,CreateEvent,62541748,Dominik-Seiler,Dominik-Seiler,,https://api.github.com/users/Dominik-Seiler,https://avatars.githubusercontent.com/u/62541748?,607237316,darthweiter/exoplanet,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,27400475736,PushEvent,8517910,LombiqBot,LombiqBot,,https://api.github.com/users/LombiqBot,https://avatars.githubusercontent.com/u/8517910?,410004573,Lombiq/TheResumeTheme,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,27400475739,PushEvent,122495445,yanocuangana,yanocuangana,,https://api.github.com/users/yanocuangana,https://avatars.githubusercontent.com/u/122495...,587937207,yanocuangana/wdd130,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,27421467397,CreateEvent,44949234,AnOnYmOuS219,AnOnYmOuS219,,https://api.github.com/users/AnOnYmOuS219,https://avatars.githubusercontent.com/u/44949234?,608266191,AnOnYmOuS219/Topics-in-AI-CSI-5180,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
996,27421467416,PushEvent,41898282,github-actions[bot],github-actions,,https://api.github.com/users/github-actions[bot],https://avatars.githubusercontent.com/u/41898282?,545057340,uday-mehtani/uday-mehtani.github.io,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
997,27421467437,PushEvent,1044413,kevinbarabash,kevinbarabash,,https://api.github.com/users/kevinbarabash,https://avatars.githubusercontent.com/u/1044413?,123198469,Khan/wonder-blocks,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
998,27421467440,IssueCommentEvent,49699333,dependabot[bot],dependabot,,https://api.github.com/users/dependabot[bot],https://avatars.githubusercontent.com/u/49699333?,532102763,behnamkhanioffical/Development-management-soft...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
def reduce_expr(expr):
    exprs = []
    while expr.expr__ is not None:
        exprs.append((expr.obj__, expr.args__, expr.kw__))
        expr = expr.expr__
    exprs.append((expr.obj__, expr.args__, expr.kw__))
    return tuple(reversed(exprs))

In [18]:
reduce_expr(e)

(('/home/brl0/e/data/gharchive/2023-03-01-0.json.gz', (), {}),
 (<function dask_obj.polars.pl_read_ndjson_fsspec(file, lines=None)>,
  (),
  {'lines': 1000}),
 (<function dask_obj.polars.unnest(df)>, (), {}))

In [12]:
reduce_expr(e)

(('/home/brl0/e/data/gharchive/2023-03-01-0.json.gz', (), {}),
 (<function dask_obj.polars.pl_read_ndjson_fsspec(file, lines=None)>,
  (),
  {'lines': 1000}),
 (<function dask_obj.polars.unnest(df)>, (), {}))